# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/busrayildirim0/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML Task Framing — Content Refresh / Declining Pages

## 1. My Lane as an ML Task

**Lane:** Content Refresh / Declining Pages

**ML task type:** Classification

I want to classify content pages according to whether their search performance is declining. The goal is to identify pages that may need content review before more search visibility is lost.

Each page will be represented by observable performance and content-related signals such as impressions, CTR, average position, content age, and word count.

The model output will be used to prioritize pages for content-refresh review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target / Proxy

**Target:** `is_declining`

A page is labeled as declining when:

`trend_direction == "down"`

Therefore:

- `1` = declining page
- `0` = not declining page

This is a practical starter target rather than a claim about the underlying cause of the decline.

The model will use only signals that could be observed before making the review decision. I will not use `trend_direction` or `trend_pct` as model features because they directly define or reveal the target and would create target leakage.

The practical question is:

> Can observable page-level signals help identify pages that are more likely to be declining, so that the content team can prioritize them for review?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success Metric

**Primary metric: Precision@K**

I will use Precision@20 and Precision@50 as the main ranking-oriented evaluation metrics.

Precision@K answers:

> Of the top K pages prioritized for review, how many are actually labeled as declining?

This metric is useful because the content team has limited review capacity. A useful system should place a high proportion of genuinely declining pages near the top of the review queue.

As a baseline, I will compare the model against a simple hand-written rule based on page staleness and visibility.

I will describe the results as observed performance on the available dataset rather than claiming that the model proves how search algorithms work.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*   Liste öğesi
*   Liste öğesi



*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd
import os

# FlyRank starter dataset
DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print(f"Dataset shape: {df.shape}")
print("One row represents one content page.")

df.head()

Dataset shape: (30000, 44)
One row represents one content page.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Declining rate:", round(df["is_declining"].mean(), 3))

df[["trend_direction", "is_declining"]].head(10)

Declining rate: 0.542


,trend_direction,is_declining
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


In [7]:
leakage_cols = ["trend_direction", "trend_pct", "content_id", "client_id"]
X = df.drop(columns=leakage_cols + ["is_declining"])
y = df["is_declining"]

In [8]:
print(f"X (Features) Shape: {X.shape}")
print(f"y (Target) Shape: {y.shape}")
print(f"Target Distribution (1 = Declining, 0 = Stable/Growing):")
print(y.value_counts(normalize=True).round(3))

display(X.head())

missing = X.isnull().sum()
missing_pct = (missing[missing > 0] / len(X)) * 100
if not missing_pct.empty:
    print("\nColumns with missing values (%):")
    print(missing_pct.round(2))
else:
    print("\nNo missing values found.")

X (Features) Shape: (30000, 40)
y (Target) Shape: (30000,)
Target Distribution (1 = Declining, 0 = Stable/Growing):
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,...,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,...,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,...,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,...,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,...,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5



Columns with missing values (%):
search_volume         8.23
competition           8.23
competition_level     8.70
cpc                   8.23
main_intent           7.91
word_count           25.66
char_count           25.66
provider_used        71.46
model_used           19.11
word_count_tier      25.66
char_count_tier      25.66
scroll_rate           0.42
dtype: float64


## 5. Why ML beats a fixed rule here

*Why a simple if-else rule fails:

Non-Linear Multi-Factor Interactions: Düşüş (trend_direction == 'down') tek bir metriğe (örneğin sadece kelime sayısı veya sadece CTR düşüklüğü) bağlı değildir. Çok yüksek CTR'a sahip bir sayfa pozisyon kaybettiği için düşüyor olabilirken, düşük pozisyondaki bir sayfa da intent uyuşmazlığından düşüyor olabilir.

Threshold Fragility (Eşik Kırılganlığı): "Eğer avg_position > 20 ve ctr < 0.02 ise düşüştedir" gibi sabit kurallar, farklı arama hacimleri (search_volume) veya içerik türleri (content_type) arasındaki segment varyasyonlarını yakalayamaz.

Non-linear Data Distributions: Aşağıdaki analizde görüleceği üzere, sabit tekil kuralların (heuristics) doğruluğu çok zayıf kalırken; ML modelleri çok boyutlu karar sınırları oluşturarak karmaşık örüntüleri yakalar.*

In [9]:
import numpy as np

simple_rule_pred = (
    (X["avg_position"] > 20) & (X["ctr"] < 0.05)
).astype(int)

rule_accuracy = (simple_rule_pred == y).mean()
print(f"Simple Fixed Rule Accuracy: {rule_accuracy:.3f}")

numeric_cols = X.select_dtypes(include=[np.number]).columns
correlations = X[numeric_cols].apply(lambda col: col.corr(y))

print("\nCorrelation of top numeric features with is_declining:")
print(correlations.sort_values(ascending=False).round(3))

Simple Fixed Rule Accuracy: 0.458

Correlation of top numeric features with is_declining:
days_with_impressions     0.190
word_count                0.090
days_since_last_update    0.081
char_count                0.072
scroll_events_90d         0.013
impressions_prev_30d      0.004
ai_traffic_pct            0.002
scroll_rate              -0.003
competition              -0.009
engagement_rate          -0.013
ai_sessions_90d          -0.013
cpc                      -0.017
impressions_90d          -0.018
search_volume            -0.019
pageviews_90d            -0.021
users_90d                -0.022
sessions_prev_30d        -0.023
sessions_90d             -0.023
days_with_sessions       -0.025
clicks_prev_30d          -0.029
avg_position             -0.029
engaged_sessions_90d     -0.035
clicks_90d               -0.040
ctr                      -0.062
sessions_last_30d        -0.064
clicks_last_30d          -0.072
impressions_last_30d     -0.094
age_tier_order           -0.156
content_age_da

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.